# 03 — Self-Supervised Pre-Training (SimCLR / SupCon)

Train the ST-GCN encoder on **ShuttleSet** skeletons using contrastive learning.
Supports two modes (ablation):
- **SimCLR** (`SSL_METHOD='simclr'`): NT-Xent loss, self-supervised (no labels needed)
- **SupCon** (`SSL_METHOD='supcon'`): Supervised contrastive loss using shot-type labels

Saved checkpoint name encodes the method: `ssl_pretrained_{method}_{layer}.pt`

In [ ]:
import os, sys, zipfile
from pathlib import Path

# ── Colab detection ───────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_ROOT   = Path('/content/drive/MyDrive/Baddiev2')
    PROJECT_PATH = Path('/content/Baddiev2')
    ZIP_PATH     = DRIVE_ROOT / 'baddiev2_colab.zip'

    if not (PROJECT_PATH / 'src').exists():
        print('Extracting project files...')
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            z.extractall(PROJECT_PATH)
        print(f'Extracted to {PROJECT_PATH}')
    else:
        print('Project already extracted.')

    sys.path.insert(0, str(PROJECT_PATH))
    os.chdir(PROJECT_PATH)

    # Override paths so outputs go to Drive (not ephemeral /content)
    import src.config as _cfg
    _cfg.MODELS_DIR            = DRIVE_ROOT / 'models'
    _cfg.RESULTS_DIR           = DRIVE_ROOT / 'results'
    _cfg.SS_SKELETONS_GDINO    = DRIVE_ROOT / 'datasets_preprocessing' / 'shuttleset_skeletons_gdino'
    _cfg.FB_SKELETONS_GDINO_V2 = DRIVE_ROOT / 'datasets_preprocessing' / 'finebadminton_skeletons_gdino_v2'
    _cfg.FB_ANNOTATIONS        = (
        DRIVE_ROOT / 'Datasets' / 'FineBadminton-dataset' / 'dataset' /
        'transformed_combined_rounds_output_en_evals_translated.json'
    )
    # ── ShuttleSet CSV annotations (shot-type labels for SupCon / §7a eval) ─
    _cfg.SS_CSV_ROOT  = DRIVE_ROOT / 'datasets' / 'ShuttleSet' / 'set'
    _cfg.SS_MATCH_CSV = _cfg.SS_CSV_ROOT / 'match.csv'
    # ── Split JSON (inside extracted zip at /content/Baddiev2) ───────────────
    _cfg.SS_SPLIT_JSON = PROJECT_PATH / 'datasets_preprocessing' / 'shuttleset_split.json'

    _cfg.MODELS_DIR.mkdir(parents=True, exist_ok=True)
    _cfg.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    print(f'Drive root   : {DRIVE_ROOT}')
    print(f'Models dir   : {_cfg.MODELS_DIR}')
    print(f'SS skeletons : {_cfg.SS_SKELETONS_GDINO}')
    print(f'FB skeletons : {_cfg.FB_SKELETONS_GDINO_V2}')
    print(f'SS CSV root  : {_cfg.SS_CSV_ROOT}')
    print(f'SS CSV exists: {_cfg.SS_CSV_ROOT.exists()}')
else:
    sys.path.insert(0, os.path.abspath('..'))
    DRIVE_ROOT = Path('..')
    print('Local run — using paths from config.py')


In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from src.config import *
from src.data.graph_builder import GraphBuilder
from src.data.dataset import ShuttleSetDataset
from src.models.stgcn_model import STGCN
from src.models.simclr_loss import (
    NTXentLoss, SupConLoss, ProjectionHead, SkeletonAugmentor
)

device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else
    'cpu'
)
print(f"Device: {device}")

## 1. Configuration

In [ ]:
cfg = get_config('ssl_pretraining')

# ── Ablation toggles ──────────────────────────────────────────────────────────
FEATURE_LAYER = 'L2'    # 'L0' | 'L1' | 'L2' | 'L3'
SSL_METHOD    = 'supcon'  # 'simclr' | 'supcon'
# ─────────────────────────────────────────────────────────────────────────────

feature_dim = FEATURE_DIMS[FEATURE_LAYER]
cfg.stgcn.in_channels = feature_dim

print(f"Feature layer : {FEATURE_LAYER} ({feature_dim} features/node)")
print(f"SSL method    : {SSL_METHOD}")
print(f"\nEncoder: ST-GCN")
print(f"  in_channels  : {cfg.stgcn.in_channels}")
print(f"  num_nodes    : {cfg.stgcn.num_nodes}")
print(f"  embedding_dim: {cfg.stgcn.embedding_dim}")
print(f"\nSSL config:")
print(f"  temperature  : {cfg.ssl.temperature}")
print(f"  epochs       : {cfg.ssl.epochs}")
print(f"  batch_size   : {cfg.ssl.batch_size}")

## 2. Build Model Components

In [ ]:
# Build graph adjacency
graph_builder = GraphBuilder(
    use_inter_player=cfg.ablation.use_inter_player,
    single_player=cfg.ablation.single_player,
)
adjacency = graph_builder.build_adjacency().to(device)
print(f"Adjacency: {adjacency.shape}")

# Build encoder
encoder = STGCN(
    in_channels=cfg.stgcn.in_channels,
    num_nodes=cfg.stgcn.num_nodes,
    adjacency=adjacency,
    num_layers=cfg.stgcn.num_layers,
    base_channels=cfg.stgcn.base_channels,
    embedding_dim=cfg.stgcn.embedding_dim,
    temporal_kernel=cfg.stgcn.temporal_kernel,
    dropout=cfg.stgcn.dropout,
).to(device)

# Projection head
projector = ProjectionHead(
    embedding_dim=cfg.stgcn.embedding_dim,
    hidden_dim=cfg.ssl.projection_hidden,
    projection_dim=cfg.ssl.projection_dim,
).to(device)

# Loss — depends on SSL_METHOD
if SSL_METHOD == 'supcon':
    contrastive_loss = SupConLoss(temperature=cfg.ssl.temperature)
    print(f"Loss: SupConLoss (uses shot-type labels as positives)")
else:
    contrastive_loss = NTXentLoss(temperature=cfg.ssl.temperature)
    print(f"Loss: NTXentLoss (self-supervised, no labels)")

augmentor = SkeletonAugmentor(
    jitter_std=cfg.ssl.jitter_std,
    mask_ratio=cfg.ssl.mask_ratio,
    speed_range=cfg.ssl.speed_range,
    rotation_range=cfg.ssl.rotation_range,
)

params = list(encoder.parameters()) + list(projector.parameters())
optimizer = optim.AdamW(params, lr=cfg.ssl.lr, weight_decay=cfg.ssl.weight_decay)

total_params = sum(p.numel() for p in encoder.parameters())
print(f"\nEncoder parameters: {total_params:,}")
print(f"Total trainable   : {sum(p.numel() for p in params):,}")

## 3. Load Data

In [ ]:
## Inspect available skeletons before loading dataset
import numpy as np
from src.config import SS_SKELETONS_GDINO

skel_root = SS_SKELETONS_GDINO
print(f"Skeleton dir: {skel_root}")
print(f"Exists: {skel_root.exists()}\n")

match_dirs = sorted([d for d in skel_root.iterdir() if d.is_dir()]) if skel_root.exists() else []
print(f"{'Match':<65} {'skeletons.npy':>14} {'frame_nums.npy':>15} {'Frames':>8}")
print("-" * 105)
total_frames = 0
for d in match_dirs:
    has_sk = (d / 'skeletons.npy').exists()
    has_fn = (d / 'frame_nums.npy').exists()
    n_frames = 0
    if has_fn:
        n_frames = len(np.load(str(d / 'frame_nums.npy')))
        total_frames += n_frames
    sk_str = f"{np.load(str(d / 'skeletons.npy')).shape}" if has_sk else "missing"
    fn_str = str(n_frames) if has_fn else "missing"
    print(f"  {d.name:<63} {sk_str:>14} {fn_str:>15} {n_frames:>8}")

print(f"\nTotal matches: {len(match_dirs)}  |  Total frames with skeletons: {total_frames:,}")


In [ ]:
from src.config import SS_SKELETONS_GDINO, SS_SPLIT_JSON

# SupCon: train split only (labels come from CSVs via ShuttleSetDataset)
# SimCLR: can use all 38 matches (no labels needed)
split_arg = 'train' if SSL_METHOD == 'supcon' else None

dataset = ShuttleSetDataset(
    skeleton_dir=SS_SKELETONS_GDINO,
    shot_window=cfg.data.shot_window,
    feature_layer=FEATURE_LAYER,
    load_shot_types=True,
    split=split_arg,
    split_json=SS_SPLIT_JSON,
)
print(f"Dataset size: {len(dataset)} samples (split={split_arg})")

# Sanity check
sample = dataset[0]
x, st = sample if isinstance(sample, tuple) else (sample, None)
has_data = x.abs().sum() > 0
print(f"Sample shape: {x.shape}, shot_type: {st}, has_data: {has_data}")

if SSL_METHOD == 'supcon':
    n_labeled = sum(1 for _ in range(min(500, len(dataset)))
                    if dataset[_][1] >= 0)
    print(f"\nShots with valid shot-type label (first 500 sample): {n_labeled}/500")
    if n_labeled == 0:
        print("WARNING: No shot-type labels found — check dataset CSV loading")

In [ ]:
def ssl_collate(batch):
    """Collate (x, shot_type) tuples."""
    xs, labels = zip(*batch)
    return torch.stack(xs), torch.tensor(labels, dtype=torch.long)

dataloader = DataLoader(
    dataset,
    batch_size=cfg.ssl.batch_size,
    shuffle=True,
    num_workers=0,   # set to 4 on Colab
    pin_memory=True,
    drop_last=True,
    collate_fn=ssl_collate,
)
print(f"Batches per epoch: {len(dataloader)}")

## 3b. Method Visualisations: SimCLR vs SupCon

Four diagrams to build and test intuition before training:

1. **Augmentation pipeline** — what two different "views" of the same shot look like as skeleton stick figures
2. **Positive-pair structure** — which cells in the 2B×2B similarity matrix act as "pull together" vs "push apart"
3. **Training signal richness** — how many positives each anchor gets per method
4. **Intra-class consistency** *(hypothesis test)* — do shots with the same label actually share similar skeletal posture? If within-row variance is high, SupCon's positive-pairing assumption may not hold and a poor result would be expected

> These cells are diagnostic only and do not affect training.

In [ ]:
# ── Viz 1: Augmentation pipeline — two views of the same shot ────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np, torch

_COCO_EDGES = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),(5,7),(7,9),(6,8),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16),
]

def _draw_player(ax, xy17, color, valid_mask, alpha=1.0):
    """Draw skeleton using pre-computed validity mask (avoids jitter corrupting zero joints)."""
    for j1, j2 in _COCO_EDGES:
        if valid_mask[j1] and valid_mask[j2]:
            ax.plot([xy17[j1,0], xy17[j2,0]], [-xy17[j1,1], -xy17[j2,1]],
                    '-', color=color, lw=2, alpha=alpha)
    for j in range(17):
        if valid_mask[j]:
            ax.scatter(xy17[j,0], -xy17[j,1], s=25, c=color, alpha=alpha, zorder=5)

def _get_xy_and_valid(orig_tensor, aug_tensor):
    """
    Returns (p0_xy, p1_xy, valid_p0, valid_p1).
    Validity is determined from the ORIGINAL skeleton — jitter moves zero joints
    to ~noise level so we can't threshold the augmented tensor reliably.
    For masking, we additionally zero-out explicitly masked joints.
    """
    orig_xy  = orig_tensor[:2, orig_tensor.shape[1] // 2, :].T.numpy()   # (34,2)
    aug_xy   = aug_tensor[:2,  aug_tensor.shape[1]  // 2, :].T.numpy()   # (34,2)

    # Valid = non-zero in ORIGINAL
    orig_valid = np.array([abs(orig_xy[j,0]) + abs(orig_xy[j,1]) > 1e-4 for j in range(34)])
    # Also zero out joints explicitly masked in the augmented version
    aug_valid  = np.array([abs(aug_xy[j,0])  + abs(aug_xy[j,1])  > 1e-4 for j in range(34)])
    valid = orig_valid & aug_valid

    return aug_xy[:17], aug_xy[17:], valid[:17], valid[17:]

# ─────────────────────────────────────────────────────────────────────────────
raw_x = None
for i in range(len(dataset)):
    x_i, _ = dataset[i]
    if x_i.abs().sum() > 0:
        raw_x = x_i; break
assert raw_x is not None

_aug_full = SkeletonAugmentor(
    jitter_std=cfg.ssl.jitter_std, mask_ratio=cfg.ssl.mask_ratio,
    speed_range=cfg.ssl.speed_range, rotation_range=cfg.ssl.rotation_range,
)
_aug_variants = {
    'Original\n(raw skeleton)': raw_x,
    'Jitter\n(Gaussian noise)':
        SkeletonAugmentor(jitter_std=0.06, mask_ratio=0, speed_range=0, rotation_range=0)(raw_x),
    'Speed perturb\n(full seq at 0.7x-1.3x)':
        SkeletonAugmentor(jitter_std=0, mask_ratio=0, speed_range=0.3, rotation_range=0)(raw_x),
    'Rotation\n(+/-25 deg)':
        SkeletonAugmentor(jitter_std=0, mask_ratio=0, speed_range=0, rotation_range=25)(raw_x),
    'Joint masking\n(40% joints zeroed)':
        SkeletonAugmentor(jitter_std=0, mask_ratio=0.40, speed_range=0, rotation_range=0)(raw_x),
    'View A\n(all augs combined)':  _aug_full(raw_x),
    'View B\n(all augs combined)':  _aug_full(raw_x),
}

fig, axes = plt.subplots(1, len(_aug_variants), figsize=(3.5 * len(_aug_variants), 5))
fig.suptitle(
    'Augmentation Pipeline — Two Contrastive Views of One Badminton Shot\n'
    'Blue = Player 1 (top court)  |  Red = Player 2 (bottom court)',
    fontsize=12, fontweight='bold', y=1.01,
)

for ax, (title, x_aug) in zip(axes, _aug_variants.items()):
    p0, p1, v0, v1 = _get_xy_and_valid(raw_x, x_aug)
    _draw_player(ax, p0, color='royalblue', valid_mask=v0)
    _draw_player(ax, p1, color='tomato',    valid_mask=v1)
    ax.set_title(title, fontsize=9)
    ax.set_aspect('equal')
    ax.axis('off')

for ax in axes[-2:]:
    for spine in ax.spines.values():
        spine.set_visible(True); spine.set_linewidth(3); spine.set_edgecolor('#2ca02c')
    ax.set_facecolor('#f0fff0')

green_patch = mpatches.Patch(facecolor='#f0fff0', edgecolor='#2ca02c', lw=2,
                              label='Encoder input pair (both views of same shot)')
fig.legend(handles=[green_patch], loc='lower center', fontsize=9, bbox_to_anchor=(0.5, -0.04))
plt.tight_layout()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(RESULTS_DIR / 'ssl_viz1_augmentation_views.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/ssl_viz1_augmentation_views.png")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Toy batch: 3 shot types, 4 samples each  =>  B=12, similarity matrix is 24x24
_NPC, _CLS = 4, ['Smash', 'Clear', 'Drive']
_B   = _NPC * len(_CLS)
_lbl = np.concatenate([np.repeat(np.arange(len(_CLS)), _NPC)] * 2)

_sc = np.zeros((2*_B, 2*_B), bool)   # SimCLR: augmentation twins only
for _i in range(_B):
    _sc[_i, _i+_B] = _sc[_i+_B, _i] = True

_sp = (_lbl[:,None] == _lbl[None,:])  # SupCon: all same-class
np.fill_diagonal(_sp, False)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 5.5))
fig.suptitle(f'Positive Pair Structure in the 2B x 2B Similarity Matrix  (B={_B})',
             fontsize=12, fontweight='bold')

for ax, title, sub, mask, col in [
    (ax0, 'SimCLR (NT-Xent)', '1 positive per anchor -- augmentation twin only', _sc, '#1f77b4'),
    (ax1, 'SupCon (Supervised Contrastive)', f'{2*_NPC-1} positives per anchor -- all same-class pairs', _sp, '#d62728'),
]:
    bg = np.ones((2*_B, 2*_B, 4))
    bg[mask] = [int(col[1:3],16)/255, int(col[3:5],16)/255, int(col[5:7],16)/255, 0.75]
    ax.imshow(bg, interpolation='none')
    ax.axhline(_B-0.5, color='black', lw=2)
    ax.axvline(_B-0.5, color='black', lw=2)
    for c in range(1, len(_CLS)):
        for off in [0, _B]:
            p = off + c*_NPC - 0.5
            ax.axhline(p, color='gray', lw=0.7, ls='--')
            ax.axvline(p, color='gray', lw=0.7, ls='--')
    tks = [off + c*_NPC + _NPC/2 - 0.5 for off in [0,_B] for c in range(len(_CLS))]
    ax.set_xticks(tks); ax.set_xticklabels(_CLS*2, fontsize=9)
    ax.set_yticks(tks); ax.set_yticklabels(_CLS*2, fontsize=9)
    ax.text(_B/2-0.5,    -2.4, 'z_i  (View 1)', ha='center', fontsize=9, color='navy', fontweight='bold')
    ax.text(_B+_B/2-0.5, -2.4, 'z_j  (View 2)', ha='center', fontsize=9, color='navy', fontweight='bold')
    ax.set_title(f'{title}\n{sub}', fontsize=10, pad=8)
    ax.text(2*_B-0.5, 2*_B-0.5, f'{int(mask.sum())} positive pairs',
            ha='right', va='bottom', fontsize=8, color=col, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', ec=col, alpha=0.9))

fig.legend(handles=[
    mpatches.Patch(fc='#1f77b4', alpha=0.75, label='Positive (SimCLR)'),
    mpatches.Patch(fc='#d62728', alpha=0.75, label='Positive (SupCon)'),
    mpatches.Patch(fc='white', ec='lightgray', label='Negative'),
], loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.06))
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'ssl_viz2_positive_pair_structure.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/ssl_viz2_positive_pair_structure.png")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Same toy batch used in Viz 2
_NPC, _CLS = 4, ['Smash', 'Clear', 'Drive']
_B   = _NPC * len(_CLS)
_lbl = np.concatenate([np.repeat(np.arange(len(_CLS)), _NPC)] * 2)

_sc_n = np.ones(2*_B, dtype=int)   # SimCLR: always 1 positive per anchor
_sp_n = np.array([int((_lbl == _lbl[i]).sum()) - 1 for i in range(2*_B)])

fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(2*_B)
w = 0.38
ax.bar(x - w/2, _sc_n, w, color='#1f77b4', alpha=0.85, label='SimCLR')
ax.bar(x + w/2, _sp_n, w, color='#d62728', alpha=0.85, label='SupCon')
ax.axvline(_B - 0.5, color='black', lw=2, ls='--', label='View 1 | View 2')

tks = [off + c*_NPC + _NPC/2 - 0.5 for off in [0,_B] for c in range(len(_CLS))]
ax.set_xticks(tks); ax.set_xticklabels(_CLS*2, fontsize=10)
ax.set_ylabel('Positives per anchor'); ax.set_ylim(0, max(_sp_n) + 2)
ax.set_title(
    'Training Signal Richness: Positives per Anchor  (SimCLR vs SupCon)\n'
    'SupCon provides more gradient signal per step for larger classes',
    fontsize=11, fontweight='bold',
)
ax.legend(fontsize=10); ax.grid(True, alpha=0.3, axis='y')

for i, (s, sp) in enumerate(zip(_sc_n, _sp_n)):
    ax.text(x[i]-w/2, s+0.1, str(s), ha='center', va='bottom', fontsize=7, color='#1f77b4')
    ax.text(x[i]+w/2, sp+0.1, str(sp), ha='center', va='bottom', fontsize=7, color='#d62728')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'ssl_viz3_training_signal.png', dpi=150, bbox_inches='tight')
plt.show()

_mean_sc = _sc_n.mean(); _mean_sp = _sp_n.mean()
print(f"SimCLR: {_mean_sc:.1f} positives per anchor (constant)")
print(f"SupCon: {_mean_sp:.1f} positives per anchor on average ({_mean_sp/_mean_sc:.1f}x more signal)")
print("Saved: results/ssl_viz3_training_signal.png")

In [ ]:
# ── Viz 4: Intra-class skeleton consistency ───────────────────────────────────
# The SupCon hypothesis is: same shot type => similar skeleton posture.
# This grid tests that assumption directly with REAL data.
#
# Layout: rows = shot types, columns = individual shot samples (mid-frame pose)
# What to look for:
#   - LOW intra-row variance  => hypothesis holds, SupCon positives are coherent
#   - HIGH intra-row variance => same label, very different poses => SupCon may
#     pull apart things that look nothing alike => noisy or failing gradients
#
# If rows look like random noise within each type, reconsider whether shot-type
# is a good proxy label for skeleton-level features.

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from collections import defaultdict
from src.config import UNIFIED_SHOT_TYPES

_COCO_EDGES = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),(5,7),(7,9),(6,8),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16),
]

def _draw_skel(ax, xy17, color, alpha=1.0):
    valid = [abs(xy17[j,0])+abs(xy17[j,1]) > 1e-4 for j in range(17)]
    for j1, j2 in _COCO_EDGES:
        if valid[j1] and valid[j2]:
            ax.plot([xy17[j1,0],xy17[j2,0]],[-xy17[j1,1],-xy17[j2,1]],'-',color=color,lw=1.8,alpha=alpha)
    for j in range(17):
        if valid[j]:
            ax.scatter(xy17[j,0],-xy17[j,1],s=18,c=color,alpha=alpha,zorder=5)

def _get_mid(x_tensor):
    xy = x_tensor[:2, x_tensor.shape[1]//2, :].T.numpy()
    return xy[:17], xy[17:]

# ── Collect samples per shot type ────────────────────────────────────────────
N_COLS   = 5   # samples per shot type to show
MIN_SAMPLES = 3  # skip types with fewer than this many non-zero samples

by_type = defaultdict(list)
for i in range(len(dataset)):
    x, st = dataset[i]
    if st >= 0 and x.abs().sum() > 0:
        by_type[st].append(x)

# Pick the most common types that have enough samples
available = [(st, xs) for st, xs in by_type.items() if len(xs) >= MIN_SAMPLES]
available.sort(key=lambda kv: -len(kv[1]))
show_types = available[:8]   # up to 8 rows

if not show_types:
    print("No labeled, non-zero samples found — run with a split that has shot-type labels.")
else:
    N_ROWS = len(show_types)
    fig = plt.figure(figsize=(N_COLS * 2.2, N_ROWS * 2.8))
    fig.suptitle(
        'Viz 4 — Intra-class Skeleton Consistency per Shot Type\n'
        'Each row = same shot-type label | Each column = a different shot instance\n'
        'Low within-row variance => SupCon positives are coherent | '
        'High variance => hypothesis at risk',
        fontsize=11, fontweight='bold', y=1.01,
    )

    gs = gridspec.GridSpec(N_ROWS, N_COLS, hspace=0.6, wspace=0.3)
    rng = np.random.RandomState(42)

    for row_i, (st, xs) in enumerate(show_types):
        # Sample N_COLS shots randomly (reproducible)
        chosen = rng.choice(len(xs), size=min(N_COLS, len(xs)), replace=False)
        label_name = UNIFIED_SHOT_TYPES[st] if st < len(UNIFIED_SHOT_TYPES) else f'type_{st}'
        label_name_disp = label_name.replace('_', '\n')

        for col_i, idx in enumerate(chosen):
            ax = fig.add_subplot(gs[row_i, col_i])
            p0, p1 = _get_mid(xs[idx])
            _draw_skel(ax, p0, color='royalblue')
            _draw_skel(ax, p1, color='tomato')
            ax.set_aspect('equal'); ax.axis('off')
            if col_i == 0:
                ax.set_ylabel(label_name_disp, fontsize=9, rotation=0,
                              labelpad=60, va='center', ha='right')
            if row_i == 0:
                ax.set_title(f'Shot {col_i+1}', fontsize=8, color='gray')

        # Fill empty columns if fewer than N_COLS samples
        for col_i in range(len(chosen), N_COLS):
            ax = fig.add_subplot(gs[row_i, col_i])
            ax.axis('off')
            ax.text(0.5, 0.5, f'only\n{len(xs)} samples', ha='center', va='center',
                    fontsize=7, color='lightgray', transform=ax.transAxes)

    # Row label header
    fig.text(0.01, 0.5, 'Shot type label', va='center', rotation='vertical',
             fontsize=10, color='navy', fontweight='bold')

    plt.savefig(RESULTS_DIR / 'ssl_viz4_intraclass_consistency.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    # Print intra-class sample counts for reference
    print(f"\nShot types shown ({N_ROWS}):")
    for st, xs in show_types:
        name = UNIFIED_SHOT_TYPES[st] if st < len(UNIFIED_SHOT_TYPES) else f'type_{st}'
        print(f"  {name:<25} {len(xs):>4} samples in dataset")
    print("\nSaved: results/ssl_viz4_intraclass_consistency.png")
    print("\nInterpretation:")
    print("  Poses look similar within a row  => SupCon hypothesis holds")
    print("  Poses look random within a row   => shot-type is a poor proxy label")

history = {'loss': [], 'epoch': []}

# Early stopping config
EARLY_STOP_PATIENCE = 10   # stop if no improvement for this many epochs
EARLY_STOP_MIN_DELTA = 1e-4
best_loss = float('inf')
no_improve_count = 0

for epoch in range(cfg.ssl.epochs):
    encoder.train()
    projector.train()

    epoch_loss = 0.0

    for x_batch, labels_batch in tqdm(dataloader, desc=f'Epoch {epoch+1}/{cfg.ssl.epochs}', leave=False):
        x = x_batch.to(device)   # (B, C, T, V)

        # Two augmented views
        x_i = torch.stack([augmentor(xi) for xi in x])
        x_j = torch.stack([augmentor(xi) for xi in x])

        h_i = encoder(x_i)   # (B, embedding_dim)
        h_j = encoder(x_j)

        z_i = projector(h_i)
        z_j = projector(h_j)

        if SSL_METHOD == 'supcon':
            valid = labels_batch >= 0
            if not valid.any():
                continue
            loss = contrastive_loss(
                z_i[valid], z_j[valid],
                labels_batch[valid].to(device)
            )
        else:  # simclr
            loss = contrastive_loss(z_i, z_j)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(dataloader)
    history['loss'].append(avg_loss)
    history['epoch'].append(epoch + 1)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d}/{cfg.ssl.epochs} | loss: {avg_loss:.4f}")

    # Early stopping check
    if avg_loss < best_loss - EARLY_STOP_MIN_DELTA:
        best_loss = avg_loss
        no_improve_count = 0
    else:
        no_improve_count += 1
        if no_improve_count >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1} (no improvement for {EARLY_STOP_PATIENCE} epochs)")
            break

print(f"Training complete. Best loss: {best_loss:.4f}")

In [ ]:
history = {'loss': [], 'epoch': []}

for epoch in range(cfg.ssl.epochs):
    encoder.train()
    projector.train()

    epoch_loss = 0.0

    for x_batch, labels_batch in tqdm(dataloader, desc=f'Epoch {epoch+1}/{cfg.ssl.epochs}', leave=False):
        x = x_batch.to(device)   # (B, C, T, V)

        # Two augmented views
        x_i = torch.stack([augmentor(xi) for xi in x])
        x_j = torch.stack([augmentor(xi) for xi in x])

        h_i = encoder(x_i)   # (B, embedding_dim)
        h_j = encoder(x_j)

        z_i = projector(h_i)
        z_j = projector(h_j)

        if SSL_METHOD == 'supcon':
            # Filter to samples with valid labels (≥0)
            valid = labels_batch >= 0
            if not valid.any():
                continue  # skip batch if no labels available
            loss = contrastive_loss(
                z_i[valid], z_j[valid],
                labels_batch[valid].to(device)
            )
        else:  # simclr
            loss = contrastive_loss(z_i, z_j)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(dataloader)
    history['loss'].append(avg_loss)
    history['epoch'].append(epoch + 1)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d}/{cfg.ssl.epochs} | loss: {avg_loss:.4f}")

print("Training complete.")

## 5. Save Weights

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

checkpoint = {
    'encoder_state_dict': encoder.state_dict(),
    'projector_state_dict': projector.state_dict(),
    'feature_layer': FEATURE_LAYER,
    'ssl_method': SSL_METHOD,
    'config': cfg,
    'history': history,
}

ckpt_path = MODELS_DIR / f'ssl_pretrained_{SSL_METHOD}_{FEATURE_LAYER}.pt'
torch.save(checkpoint, ckpt_path)
print(f"Saved: {ckpt_path}")

## 5b. Train All Feature Layers (L0–L3)
Run this cell to generate checkpoints for all feature layers in one go.
Skips any layer whose `ssl_pretrained_<LAYER>.pt` already exists.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Train All Feature Layers (L0 → L3) in one run.
# Skips any layer whose checkpoint already exists in models/.
# Each layer gets a fresh encoder, projector, aux_head, and optimizer.
# ══════════════════════════════════════════════════════════════════════════════
from src.config import SS_SKELETONS_GDINO

cfg = get_config('ssl_pretraining')

for FEATURE_LAYER in ['L0', 'L1', 'L2', 'L3']:
    save_path = MODELS_DIR / f'ssl_pretrained_{FEATURE_LAYER}.pt'
    if save_path.exists():
        print(f"\n[SKIP] {FEATURE_LAYER} — checkpoint already exists at {save_path}")
        continue

    print(f"\n{'='*60}")
    print(f"Training SSL — Feature Layer: {FEATURE_LAYER}")
    print('='*60)

    feature_dim = FEATURE_DIMS[FEATURE_LAYER]
    cfg.stgcn.in_channels = feature_dim

    # ── Dataset ───────────────────────────────────────────────────────────────
    _dataset = ShuttleSetDataset(
        skeleton_dir=SS_SKELETONS_GDINO,
        shot_window=cfg.data.shot_window,
        feature_layer=FEATURE_LAYER,
        load_shot_types=True,
    )
    print(f"Dataset: {len(_dataset)} samples, mode={_dataset._mode}")

    _dataloader = DataLoader(
        _dataset,
        batch_size=cfg.ssl.batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
        drop_last=True,
        collate_fn=ssl_collate,
    )

    # ── Model ─────────────────────────────────────────────────────────────────
    graph_builder = GraphBuilder(
        use_inter_player=cfg.ablation.use_inter_player,
        single_player=cfg.ablation.single_player,
    )
    adjacency = graph_builder.build_adjacency().to(device)

    _encoder = STGCN(
        in_channels=cfg.stgcn.in_channels,
        num_nodes=cfg.stgcn.num_nodes,
        adjacency=adjacency,
        num_layers=cfg.stgcn.num_layers,
        base_channels=cfg.stgcn.base_channels,
        embedding_dim=cfg.stgcn.embedding_dim,
        temporal_kernel=cfg.stgcn.temporal_kernel,
        dropout=cfg.stgcn.dropout,
    ).to(device)
    _projector = ProjectionHead(
        embedding_dim=cfg.stgcn.embedding_dim,
        hidden_dim=cfg.ssl.projection_hidden,
        projection_dim=cfg.ssl.projection_dim,
    ).to(device)
    _aux_head = AuxiliaryShotTypeHead(
        embedding_dim=cfg.stgcn.embedding_dim,
        num_shot_types=cfg.ssl.num_shot_types,
    ).to(device)

    _contrastive_loss = SupConLoss(temperature=cfg.ssl.temperature) if SSL_METHOD == 'supcon' else NTXentLoss(temperature=cfg.ssl.temperature)
    _aux_criterion = nn.CrossEntropyLoss()
    _augmentor = SkeletonAugmentor(
        jitter_std=cfg.ssl.jitter_std,
        mask_ratio=cfg.ssl.mask_ratio,
        speed_range=cfg.ssl.speed_range,
        rotation_range=cfg.ssl.rotation_range,
    )
    _optimizer = optim.AdamW(
        list(_encoder.parameters()) + list(_projector.parameters()) + list(_aux_head.parameters()),
        lr=cfg.ssl.lr, weight_decay=cfg.ssl.weight_decay,
    )

    # ── Training ──────────────────────────────────────────────────────────────
    _history = {'loss': [], 'contrastive_loss': [], 'aux_loss': [], 'epoch': []}

    for epoch in range(cfg.ssl.epochs):
        _encoder.train(); _projector.train(); _    epoch_loss = epoch_cl = epoch_aux = 0.0

        for x_batch, labels_batch in _dataloader:
            x = x_batch.to(device)
            x_i = torch.stack([_augmentor(xi) for xi in x])
            x_j = torch.stack([_augmentor(xi) for xi in x])

            h_i = _encoder(x_i)
            h_j = _encoder(x_j)
            z_i = _projector(h_i)
            z_j = _projector(h_j)
            cl_loss = _contrastive_loss(z_i, z_j)

            total_loss = cl_loss
            aux_loss_val = 0.0
            if cfg.ablation.use_auxiliary_task:
                valid_mask = labels_batch >= 0
                if valid_mask.any():
                    aux_logits = _aux_head(h_i[valid_mask])
                    aux_loss = _aux_criterion(aux_logits, labels_batch[valid_mask].to(device))
                    total_loss = cl_loss + cfg.ssl.auxiliary_weight * aux_loss
                    aux_loss_val = aux_loss.item()

            _optimizer.zero_grad()
            total_loss.backward()
            _optimizer.step()

            epoch_loss += total_loss.item()
            epoch_cl += cl_loss.item()
            epoch_aux += aux_loss_val

        n = len(_dataloader)
        _history['loss'].append(epoch_loss / n)
        _history['contrastive_loss'].append(epoch_cl / n)
        _history['aux_loss'].append(epoch_aux / n)
        _history['epoch'].append(epoch + 1)

        if (epoch + 1) % 10 == 0:
            print(f"  [{FEATURE_LAYER}] Epoch {epoch+1}/{cfg.ssl.epochs} — "
                  f"Loss: {epoch_loss/n:.4f} (CL: {epoch_cl/n:.4f}, Aux: {epoch_aux/n:.4f})")

    # ── Save ──────────────────────────────────────────────────────────────────
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    torch.save({
        'encoder_state_dict': _encoder.state_dict(),
        'projector_state_dict': _projector.state_dict(),
        'aux_head_state_dict': _aux_head.state_dict(),
        'optimizer_state_dict': _optimizer.state_dict(),
        'feature_layer': FEATURE_LAYER,
        'feature_dim': feature_dim,
        'config': cfg,
        'history': _history,
    }, save_path)
    print(f"  Saved → {save_path}")

print("\nAll feature layers complete.")


## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(8, 5))
axes.plot(history['epoch'], history['loss'], label=f'{SSL_METHOD} loss')
axes.set_xlabel('Epoch')
axes.set_ylabel('Loss')
axes.set_title(f'SSL Pre-training Loss ({SSL_METHOD.upper()}, {FEATURE_LAYER})')
axes.legend()
axes.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(MODELS_DIR / f'ssl_loss_{SSL_METHOD}_{FEATURE_LAYER}.png', dpi=150)
plt.show()

## 7. Proxy Evaluation

Two evaluations to check representation quality before running few-shot:

**7a. Shot-type classification on SS val set** — freeze encoder, train linear head on SS train,
evaluate on 6 held-out val matches. This is the direct proxy for SupCon quality.

**7b. Linear probe on FineBadminton strategy labels** — cross-validated logistic regression.
Sanity-check that the representation transfers to the downstream task.

In [ ]:
## 7a. Shot-Type Classification on SS Val Set
from src.data.dataset import ShuttleSetDataset
from src.config import SS_SKELETONS_GDINO, SS_SPLIT_JSON
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report
import numpy as np

def extract_embeddings(ds, enc, dev):
    enc.eval()
    embs, labs = [], []
    with torch.no_grad():
        for i in range(len(ds)):
            sample = ds[i]
            x, y = sample if isinstance(sample, tuple) else (sample, -1)
            if y < 0:
                continue
            emb = enc(x.unsqueeze(0).to(dev)).cpu().numpy()[0]
            embs.append(emb)
            labs.append(y)
    return np.array(embs), np.array(labs)

# SS train split — fit classifier
ss_train = ShuttleSetDataset(
    skeleton_dir=SS_SKELETONS_GDINO,
    shot_window=cfg.data.shot_window,
    feature_layer=FEATURE_LAYER,
    load_shot_types=True,
    split='train',
    split_json=SS_SPLIT_JSON,
)
X_train, y_train = extract_embeddings(ss_train, encoder, device)
print(f"SS train embeddings: {X_train.shape}  ({len(set(y_train))} classes)")

# SS val split — evaluate
ss_val = ShuttleSetDataset(
    skeleton_dir=SS_SKELETONS_GDINO,
    shot_window=cfg.data.shot_window,
    feature_layer=FEATURE_LAYER,
    load_shot_types=True,
    split='val',
    split_json=SS_SPLIT_JSON,
)
X_val, y_val = extract_embeddings(ss_val, encoder, device)
print(f"SS val   embeddings: {X_val.shape}")

clf = LogisticRegression(max_iter=1000, C=1.0)
clf.fit(X_train, y_train)
preds = clf.predict(X_val)

macro_f1 = f1_score(y_val, preds, average='macro')
print(f"\nSS Val Shot-Type Macro-F1: {macro_f1:.3f}")
print(classification_report(y_val, preds, zero_division=0))
# Save shot-type classifier to Drive for use at inference
import joblib
shot_clf_path = MODELS_DIR / f'shot_type_clf_{SSL_METHOD}_{FEATURE_LAYER}.joblib'
joblib.dump(clf, shot_clf_path)
print(f'\nShot-type classifier saved: {shot_clf_path}')
print(f'Classes: {clf.classes_}')


In [ ]:
## 7b. Linear Probe on FineBadminton Strategy Labels
from src.data.dataset import FineBadmintonDataset
from sklearn.model_selection import StratifiedKFold

fb_ds = FineBadmintonDataset(feature_layer=FEATURE_LAYER)
print(f"FineBadminton: {len(fb_ds)} samples")

fb_embs, fb_labs = extract_embeddings(fb_ds, encoder, device)
print(f"Embeddings: {fb_embs.shape}")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_f1s = []
for train_idx, test_idx in skf.split(fb_embs, fb_labs):
    clf = LogisticRegression(max_iter=1000, C=1.0)
    clf.fit(fb_embs[train_idx], fb_labs[train_idx])
    preds = clf.predict(fb_embs[test_idx])
    fold_f1s.append(f1_score(fb_labs[test_idx], preds, average='macro'))

print(f"\nFB Strategy Linear Probe Macro-F1: {np.mean(fold_f1s):.3f} ± {np.std(fold_f1s):.3f}")
print(f"Per-fold: {[f'{f:.3f}' for f in fold_f1s]}")

In [ ]:
## 5c. Train Both SSL Methods — L2 (Recommended for Colab)
# Trains simclr then supcon for L2 in one run.
# Skips any method whose checkpoint already exists on Drive.

def _train_ssl_method(ssl_method, layer='L2'):
    from src.config import get_config, FEATURE_DIMS, MODELS_DIR, SS_SKELETONS_GDINO, SS_SPLIT_JSON
    from src.data.graph_builder import GraphBuilder
    from src.data.dataset import ShuttleSetDataset
    from src.models.stgcn_model import STGCN
    from src.models.simclr_loss import NTXentLoss, SupConLoss, ProjectionHead, SkeletonAugmentor
    import torch.optim as optim
    from torch.utils.data import DataLoader
    from tqdm import tqdm

    cfg = get_config('ssl_pretraining')
    cfg.stgcn.in_channels = FEATURE_DIMS[layer]

    adj = GraphBuilder(
        use_inter_player=cfg.ablation.use_inter_player,
        single_player=cfg.ablation.single_player,
    ).build_adjacency().to(device)

    encoder = STGCN(
        in_channels=cfg.stgcn.in_channels, num_nodes=cfg.stgcn.num_nodes, adjacency=adj,
        num_layers=cfg.stgcn.num_layers, base_channels=cfg.stgcn.base_channels,
        embedding_dim=cfg.stgcn.embedding_dim, temporal_kernel=cfg.stgcn.temporal_kernel,
        dropout=cfg.stgcn.dropout,
    ).to(device)
    projector = ProjectionHead(
        cfg.stgcn.embedding_dim, cfg.ssl.projection_hidden, cfg.ssl.projection_dim
    ).to(device)

    loss_fn = (
        SupConLoss(cfg.ssl.temperature) if ssl_method == 'supcon'
        else NTXentLoss(cfg.ssl.temperature)
    )
    aug = SkeletonAugmentor(
        jitter_std=cfg.ssl.jitter_std, mask_ratio=cfg.ssl.mask_ratio,
        speed_range=cfg.ssl.speed_range, rotation_range=cfg.ssl.rotation_range,
    )
    opt = optim.AdamW(
        list(encoder.parameters()) + list(projector.parameters()),
        lr=cfg.ssl.lr, weight_decay=cfg.ssl.weight_decay,
    )

    split = 'train' if ssl_method == 'supcon' else None
    ds = ShuttleSetDataset(
        skeleton_dir=SS_SKELETONS_GDINO, shot_window=cfg.data.shot_window,
        feature_layer=layer, load_shot_types=True, split=split, split_json=SS_SPLIT_JSON,
    )
    loader = DataLoader(
        ds, batch_size=cfg.ssl.batch_size, shuffle=True, num_workers=2, drop_last=True,
        collate_fn=lambda b: (torch.stack([x[0] for x in b]),
                               torch.tensor([x[1] for x in b], dtype=torch.long)),
    )

    print('\n' + '='*55)
    print(f'Training {ssl_method.upper()} | {layer} | {len(ds)} samples | {cfg.ssl.epochs} epochs')
    history = []
    for epoch in range(cfg.ssl.epochs):
        encoder.train(); projector.train()
        epoch_loss = 0.0
        for xb, lb in tqdm(loader, desc=f'Ep {epoch+1}/{cfg.ssl.epochs}', leave=False):
            xb = xb.to(device)
            zi = projector(encoder(torch.stack([aug(x) for x in xb])))
            zj = projector(encoder(torch.stack([aug(x) for x in xb])))
            if ssl_method == 'supcon':
                valid = lb >= 0
                if not valid.any(): continue
                loss = loss_fn(zi[valid], zj[valid], lb[valid].to(device))
            else:
                loss = loss_fn(zi, zj)
            opt.zero_grad(); loss.backward(); opt.step()
            epoch_loss += loss.item()
        avg = epoch_loss / len(loader)
        history.append(avg)
        if (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1:3d} | loss: {avg:.4f}')

    ckpt = MODELS_DIR / f'ssl_pretrained_{ssl_method}_{layer}.pt'
    torch.save({'encoder_state_dict': encoder.state_dict(),
                'projector_state_dict': projector.state_dict(),
                'feature_layer': layer, 'ssl_method': ssl_method,
                'history': history}, ckpt)
    print(f'  Saved to Drive: {ckpt}')
    return history


for method in ['simclr', 'supcon']:
    ckpt = MODELS_DIR / f'ssl_pretrained_{method}_L2.pt'
    if ckpt.exists():
        print(f'Skipping {method} — checkpoint exists: {ckpt.name}')
    else:
        _train_ssl_method(method, 'L2')

print('\nAll SSL checkpoints ready on Drive.')
